# Colab Training & Evaluation Setup
Run this notebook on Google Colab to prepare the repository, install dependencies, and execute object detection training and evaluation.


In [1]:
# Install required Python packages
!pip install -q torch torchvision torchaudio torchmetrics pycocotools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.7 MB/s eta 0:00:0000:01


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!ls "/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO"

annotations  test2017  train2017  val2017


In [4]:
%cd /content

!rm -rf vehicle-damage-triage
!git clone https://github.com/AymanLakhnati/AI-vehicule-damage-triage vehicle-damage-triage

%cd /content/vehicle-damage-triage

/content
Cloning into 'vehicle-damage-triage'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 207 (delta 15), reused 205 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (207/207), 37.02 MiB | 20.50 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/vehicle-damage-triage


In [5]:
!ls
!ls src

colab_setup.ipynb	     patch_colab_notebook.py  src
models			     Project_Specs.md	      train_detector_log.txt
out			     readme.md		      write_colab_ipynb.py
patch_colab_eval.py	     reports
patch_colab_notebook_fix.py  requirments.txt
baseline_model.py		evaluate_cardd_thresholded.py
cardd_audit.py			evaluate_resnet_finetuned.py
cardd_class_weights.py		evaluate_resnet.py
cardd_dataloaders.py		import_cardd.py
cardd_dataset.py		optimize_cardd_thresholds.py
cardd_detection_dataset.py	resnet_model.py
cardd_detector.py		split_dataset.py
cardd_error_analysis.py		test_dataset.py
cardd_gradcam.py		test_detection_training_batch.py
cardd_model.py			train_baseline.py
cardd_visualize_annotations.py	train_cardd_detector.py
data_audit.py			train_cardd_finetune.py
dataloaders.py			train_cardd.py
dataset.py			train_resnet_finetune.py
download_dataset.py		train_resnet.py
evaluate_baseline.py		transforms.py
evaluate_cardd_detector.py	visualize_dataset.py
evaluate_cardd_finetuned.py	visualize_detectio

In [6]:
!pip install -q pycocotools

In [7]:
import torch
import torchvision
import pycocotools

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
TorchVision: 0.26.0+cu128
CUDA: True


In [8]:
from pathlib import Path

CARDD_DRIVE = Path(
    "/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO"
)

print("CarDD exists:", CARDD_DRIVE.exists())

print(
    "Train annotations:",
    (CARDD_DRIVE / "annotations" / "instances_train2017.json").exists()
)

print(
    "Validation annotations:",
    (CARDD_DRIVE / "annotations" / "instances_val2017.json").exists()
)

print(
    "Train images:",
    (CARDD_DRIVE / "train2017").exists()
)

CarDD exists: True
Train annotations: True
Validation annotations: True
Train images: True


In [9]:
!mkdir -p "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release"

!rm -rf "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO"

!ln -s \
"/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO" \
"/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO"

In [10]:
!ls "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO/annotations"

image_info.xlsx		 instances_train2017.json
instances_test2017.json  instances_val2017.json


In [11]:
from pathlib import Path

root = Path(
    "/content/vehicle-damage-triage/data/raw/cardd/CarDD_release/CarDD_COCO"
)

print("Train images:", len(list((root / "train2017").glob("*.jpg"))))
print("Val images:", len(list((root / "val2017").glob("*.jpg"))))
print("Test images:", len(list((root / "test2017").glob("*.jpg"))))

Train images: 2816
Val images: 810
Test images: 374


In [12]:
from pathlib import Path

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/vehicle-damage-triage-models"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Permanent checkpoint directory:")
print(CHECKPOINT_DIR)

print("\nExisting files:")
for file in CHECKPOINT_DIR.glob("*"):
    print(file.name)

Permanent checkpoint directory:
/content/drive/MyDrive/vehicle-damage-triage-models

Existing files:


In [13]:
!grep -n "vehicle-damage-triage-models" src/train_cardd_detector.py

47:    "/content/drive/MyDrive/vehicle-damage-triage-models"


In [14]:
!grep -n "optimizer_state_dict" src/train_cardd_detector.py

131:            "optimizer_state_dict": optimizer.state_dict(),


In [15]:
import torch
from pathlib import Path

print("Working directory:")
!pwd

print("\nGPU:")
print(torch.cuda.get_device_name(0))

print("\nDataset:")
dataset_root = Path(
    "data/raw/cardd/CarDD_release/CarDD_COCO"
)
print(dataset_root.resolve())
print("Exists:", dataset_root.exists())

print("\nCheckpoint destination:")
checkpoint_root = Path(
    "/content/drive/MyDrive/vehicle-damage-triage-models"
)
print(checkpoint_root)
print("Exists:", checkpoint_root.exists())

Working directory:
/content/vehicle-damage-triage

GPU:
Tesla T4

Dataset:
/content/drive/MyDrive/AI_Datasets/CarDD_release/CarDD_COCO
Exists: True

Checkpoint destination:
/content/drive/MyDrive/vehicle-damage-triage-models
Exists: True


In [ ]:
!python -u src/train_cardd_detector.py --epochs 5 --batch-size 2

Using device: cuda
Checkpoints will be saved to: /content/drive/MyDrive/vehicle-damage-triage-models
GPU: Tesla T4
Training images: 2816
Validation images: 810
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100% 160M/160M [00:01<00:00, 132MB/s] 

Epoch 1/5
total_loss=0.3847
classifier_loss=0.1721
box_reg_loss=0.1484
objectness_loss=0.0376
rpn_box_loss=0.0266
Saved model permanently to: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch1.pth
Saved training state to: /content/drive/MyDrive/vehicle-damage-triage-models/cardd_detector_epoch1_training.pth
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.19s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=2.04s).
Accumulating evaluation results...
DONE (t=0.55s).
